# 07 · Production hybrid NIDS

This notebook mirrors the canonical production pipeline in the repo. The real logic lives in the Python engine files, and this notebook follows the same order:

1. Engine A: Monday-benign anomaly detection
2. Engine B: anomaly clustering
3. Engine C: day specialists
4. Section 5: temporal generalization

The owner entry point is the project launcher at `run_hybrid_nids.py`.


In [ ]:
from pathlib import Path
root = Path.cwd()
repo_root = root if (root / 'production_hybrid_nids').exists() else root.parent
PROJECT_DIR = repo_root / 'production_hybrid_nids'
print('repo_root =', repo_root)
print('PROJECT_DIR =', PROJECT_DIR)
print('exists =', PROJECT_DIR.exists())
print('core modules =', sorted(p.name for p in PROJECT_DIR.iterdir()))


## Architecture summary

The project is not a notebook-only pipeline. It is a hybrid NIDS workflow built around four production engines:

- Engine A trains a tabular FT-Transformer autoencoder on Monday benign traffic only.
- Engine B clusters anomalous flows flagged by the autoencoder into attack families.
- Engine C trains day-specific specialists with train-only feature selection and scaling.
- Section 5 tests temporal generalization with chronological train/test splits and concept drift analysis.

This notebook follows that exact order.


In [ ]:
from production_hybrid_nids.engine_a_flow_autoencoder import EngineAFlowTrainer

trainer = EngineAFlowTrainer(PROJECT_DIR)
model, thresholds = trainer.train_or_load(force=False)
print('trained model type:', type(model).__name__)
print('threshold keys:', sorted(thresholds.keys()))
for level, value in thresholds['mean_mse'].items():
    print(f'mean_mse[{level}] = {value:.6f}')


## Step 1: Engine A in production

This stage mirrors the actual autoencoder training and threshold calibration logic. It uses Monday benign traffic, fits the model only on Monday, and then thresholds on Monday validation reconstruction error.


In [ ]:
from production_hybrid_nids.engine_b_clustering import ClusteringEngine

engine_b = ClusteringEngine(PROJECT_DIR)
embeddings, raw_features, metadata = engine_b.extract_anomalous_pool(
    model,
    thresholds,
    days=('Thursday', 'Friday'),
    threshold_level='p95',
)
print('anomaly embedding shape =', embeddings.shape)
print('raw feature shape =', raw_features.shape)
print('metadata labels =', metadata['Label'].value_counts().to_dict())
cluster_report = engine_b.run_clustering(embeddings, raw_features, metadata)
print('cluster summary keys =', sorted(cluster_report.keys()))


## Step 2: Engine B - clustering anomalous flows

The anomaly pool extracted by Engine A is transformed into bottleneck embeddings and then clustered into attack families. This step is intentionally unsupervised and is the family-discovery layer of the pipeline.


In [ ]:
from production_hybrid_nids.engine_c_specialists import SpecialistEngine

engine_c = SpecialistEngine(PROJECT_DIR)
# Uncomment a single day if you want to run one specialist only.
# metrics = engine_c.train_specialist('Tuesday', force=False)
metrics_all = engine_c.run_all()
print('specialist summary keys =', sorted(metrics_all.keys()) if isinstance(metrics_all, dict) else type(metrics_all).__name__)


## Step 3: Engine C - day specialists

Each day specialist uses train-split-only medians, Random Forest + Mutual Information ranking, and a hybrid FT-Transformer + TCN architecture. The key rule is that all feature selection and scaling are fit only on the train split and then frozen at inference time.


In [ ]:
from production_hybrid_nids.section5_temporal_generalization import TemporalGeneralizationEngine

engine_5 = TemporalGeneralizationEngine(PROJECT_DIR)
results = engine_5.run_all()
print('section5 result type =', type(results).__name__)
print('section5 keys =', sorted(results.keys()) if isinstance(results, dict) else results)


## Step 4: Section 5 - temporal generalization

This stage splits each day chronologically into a 70/30 train/test experiment, measures concept drift, and tracks performance degradation over time. It is the final check on whether the model remains stable under realistic traffic evolution.


In [ ]:
from run_hybrid_nids import run_full_pipeline

print('Running the full canonical pipeline using the project launcher...')
# run_full_pipeline(force=False)
print('Use the project launcher from a terminal with: python run_hybrid_nids.py --mode all')


## Final note

The Python pipeline files are the source of truth. The notebooks are explanatory mirrors for Colab-style walkthroughs, but the real execution logic remains in the production scripts and the single project launcher.
